In [1]:
import pandas as pd
import os
import json
# dataset_list = ['banking', 'stackoverflow', 'clinc', 'hwu', 'mcid', 'ecdt']
dataset_list = ['medical', 'ele']


In [2]:
data_statics = {}
for dataset_name in dataset_list:
    ans = {}
    data_statics[dataset_name] = {}
    for split in ['train', 'dev', 'test']:
        data_statics[dataset_name][split] = {}
        df = pd.read_csv(f'{dataset_name}/{split}.tsv', sep='\t')
        df.columns = ['text', 'label']
        ans[split] = df
    df = pd.concat([ans[i] for i in ans])
    os.makedirs(f'{dataset_name}/label', exist_ok=True)
    pd.DataFrame(df['label'].unique()).to_csv(f'{dataset_name}/label/label.list', index=None, header=None)

In [3]:
for dataset_name in dataset_list:
    for rate in [0.25, 0.5, 0.75]:
        label_list = pd.read_csv(f'{dataset_name}/label/label.list', header=None)
        known_label_list = label_list.sample(round(rate * len(label_list)))
        known_label_list.to_csv(f'{dataset_name}/label/label_{rate}.list', index=None, header=None)

In [4]:
import pandas as pd
import os
import json
import random
for dataset_name in dataset_list:
    for mode in ['train', 'dev']:
        df = pd.read_csv(f'{dataset_name}/{mode}.tsv', sep='\t')
        for labeled_ratio in [0.1]:    
            df_group = df.groupby('label').agg(list)
            df_group['text'] = df_group['text'].apply(lambda x: random.sample(x, max(round(len(x) * labeled_ratio), 1)))
            df_group = df_group.reset_index().explode('text')[['text', 'label']]
            os.makedirs(f'{dataset_name}/labeled_data', exist_ok=True)
            df_group.to_csv(f'{dataset_name}/labeled_data/{mode}_{labeled_ratio}.tsv', sep='\t', index=None)